# Forest AGB Temporal Stability — Global Tropical Forest

**Save location (local):** `D:\Rahman_MM_30062026\JC_STEM\DESK_computer\Data_folder\VS_code\Project_forest_stability\Copy_of_Mangrove_productivity.ipynb`

Python notebook version of `agb_stability_tropical_forest.js`

**Metric:** `stability = mean(AGB) / SD(AGB)` (higher = more temporally stable)

**Data:** CTREES Global AGB 100 m (`projects/sat-io/open-datasets/CTREES-GLOBAL-AGB-100M`)

**Masks (all required):**
1. WWF tropical forest biomes
2. ESA WorldCover v200 — tree cover or mangroves, exclude permanent water
3. Hansen GFC — ≥30% tree cover in 2000, no canopy loss through 2023
4. JRC Global Surface Water — occurrence < 10%
5. Mean AGB ≥ 10 Mg/ha

In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
import ee
import geemap

# Colab: authenticate if needed
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

## CONFIG

In [ ]:
START_DATE = '2000-01-01'
END_DATE   = '2025-12-31'

TREE_COVER_MIN_PCT = 30       # Hansen treecover2000 threshold
WATER_OCCURRENCE_MAX = 10     # JRC occurrence % above which pixel is excluded
MIN_MEAN_AGB = 10             # Mg/ha; drop scrub/grass misclassified as forest

MAP_CENTER = [-62, -4]
MAP_ZOOM   = 4

## CTREES AGB — load, rescale, temporal stack

In [ ]:
collection = ee.ImageCollection('projects/sat-io/open-datasets/CTREES-GLOBAL-AGB-100M')

def rescale_agb(image):
    scaled = image.multiply(image.getNumber('agb_scale_factor'))
    return scaled.updateMask(scaled.gt(0)).copyProperties(image, image.propertyNames())

filtered = collection.filterDate(START_DATE, END_DATE).map(rescale_agb)

mean_agb = filtered.select('agb').mean().rename('agb_mean')
sd_agb   = filtered.select('agb').reduce(ee.Reducer.stdDev()).rename('agb_sd')

# Stability = mean / SD  (inverse coefficient of variation; higher = more stable)
stability = (
    mean_agb
    .divide(sd_agb)
    .rename('agb_stability')
    .updateMask(sd_agb.gt(0))  # SD = 0 → undefined
)

## MASK 1 — WWF tropical forest biomes

In [ ]:
ecoregions = ee.FeatureCollection('RESOLVE/ECOREGIONS/2017')

tropical_forests = ecoregions.filter(ee.Filter.inList('BIOME_NAME', [
    'Tropical & Subtropical Moist Broadleaf Forests',
    'Tropical & Subtropical Dry Broadleaf Forests',
    'Tropical & Subtropical Coniferous Forests'
]))

tropical_biome_mask = ee.Image().paint(tropical_forests, 1).selfMask()

## MASK 2 — ESA WorldCover: forest classes, exclude water

In [ ]:
world_cover = ee.Image('ESA/WorldCover/v200').select('Map')

# 10 = Tree cover, 95 = Mangroves
wc_forest = world_cover.eq(10).Or(world_cover.eq(95))
wc_not_water = world_cover.neq(80)  # 80 = Permanent water bodies

world_cover_mask = wc_forest.And(wc_not_water)

## MASK 3 — Hansen GFC: persistent forest (no loss 2000–2023)

In [ ]:
hansen = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
hansen_year = 2023  # last loss year in this Hansen version

forest_2000 = hansen.select('treecover2000').gte(TREE_COVER_MIN_PCT)
no_loss = hansen.select('lossyear').eq(0)
stable_forest_mask = forest_2000.And(no_loss)

## MASK 4 — JRC Global Surface Water

In [ ]:
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
not_water_mask = jrc_water.lt(WATER_OCCURRENCE_MAX)

## COMBINED ANALYSIS MASK

In [ ]:
analysis_mask = (
    tropical_biome_mask
    .And(world_cover_mask)
    .And(stable_forest_mask)
    .And(not_water_mask)
    .And(mean_agb.gte(MIN_MEAN_AGB))
)

stability_masked = stability.updateMask(analysis_mask)
mean_agb_masked   = mean_agb.updateMask(analysis_mask)
sd_agb_masked     = sd_agb.updateMask(analysis_mask)

## VISUALIZATION

In [ ]:
# Mean/SD: low = variable biomass, high = stable biomass
# Use percentiles over tropical masked area for sensible stretch
stability_stats = stability_masked.reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=tropical_forests.geometry(),
    scale=1000,
    maxPixels=int(1e13),
    bestEffort=True
)

stability_vis = {
    'min': stability_stats.get('agb_stability_p2'),
    'max': stability_stats.get('agb_stability_p98'),
    'palette': [
        '#253494',  # unstable (low mean/SD)
        '#2c7fb8',
        '#41b6c4',
        '#a1dab4',
        '#ffffcc'   # stable (high mean/SD)
    ]
}

mean_vis = {'min': 0, 'max': 200, 'palette': ['#f7fcf5', '#00441b']}
sd_vis   = {'min': 0, 'max': 50,  'palette': ['#fff5f0', '#67000d']}

Map = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM)

Map.addLayer(
    stability_masked,
    stability_vis,
    'AGB Stability (mean/SD) — stable tropical forest'
)

# Toggle off by default — useful for QA
Map.addLayer(mean_agb_masked, mean_vis, 'Mean AGB', shown=False)
Map.addLayer(sd_agb_masked, sd_vis, 'SD AGB', shown=False)
Map.addLayer(analysis_mask.selfMask(), {'palette': ['#2d6a4f']}, 'Analysis mask', shown=False)

Map

## OPTIONAL — Export to Google Drive

In [ ]:
# Uncomment and set region to export

# export_region = ee.Geometry.Rectangle([-80, -25, -35, 15])  # Amazon example

# task = ee.batch.Export.image.toDrive(
#     image=stability_masked.float(),
#     description='AGB_stability_tropical_stable_forest',
#     folder='GEE_exports',
#     region=export_region,
#     scale=100,
#     maxPixels=int(1e13)
# )
# task.start()
# print('Export started:', task.id)

## OPTIONAL — Zonal summary by ecoregion

In [ ]:
# Uncomment to compute mean stability per ecoregion

# zonal_stats = stability_masked.reduceRegions(
#     collection=tropical_forests,
#     reducer=ee.Reducer.mean().combine(
#         reducer2=ee.Reducer.stdDev(),
#         sharedInputs=True
#     ),
#     scale=1000
# )

# geemap.ee_to_df(zonal_stats.limit(10))